# 17 · Preparar los juicios de pertinencia

Unión de los primeros 20 resultados de **BM25, 11 densos y 11 híbridos**, agrupando ES/EN por necesidad. Se añaden 10 artículos fuera de esa unión por necesidad, con semilla fija. **No se asigna pertinencia automáticamente.**

Guarda esta libreta en `07_RAG/Pruebas_RAG/`. Usa pandas y la biblioteca estándar; no carga modelos ni índices. Produce únicamente `conjunto_pertinencia_v1.json` y conserva las anotaciones existentes al reejecutarse.

Las rúbricas se copian del CSV revisado. Los puntajes y las posiciones no se muestran para anotar; las observaciones anteriores con artículos orientadores quedan separadas en `control_no_mostrar_al_evaluador`.

## 1. Entradas

In [1]:
import hashlib, json, math, random
from pathlib import Path
from collections import defaultdict
import pandas as pd

K_POOL = 20
EXTRAS_POR_NECESIDAD = 10
SEMILLA = 2026
MODELOS = {"bge_m3", "qwen3_0_6b", "qwen3_4b", "harrier_0_6b", "nemotron3_1b",
           "granite_311m_r2", "pplx_0_6b", "arctic_l_v2", "embeddinggemma2_text",
           "qwen3_vl_2b_text", "pplx_4b_bf16"}

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/Pruebas_RAG/hibrida_rrf_resultados_v1.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta dentro de Tesis_Multimodelo.")
CARPETA = REPO / "07_RAG/Pruebas_RAG"
ENTRADAS = {"catalogo": REPO / "06_grafos/catalogo_articulos.csv",
            "consultas": REPO / "07_RAG/consultas_evaluacion_v1.csv",
            "bm25": CARPETA / "bm25_resultados_v1.json",
            "densa": CARPETA / "densa_resultados_v1.json",
            "hibrida": CARPETA / "hibrida_rrf_resultados_v1.json"}
SALIDA = CARPETA / "conjunto_pertinencia_v1.json"

def verificar(ok, mensaje):
    if not ok:
        raise ValueError(mensaje)

def sha_json(obj):
    raw = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def sha_archivo(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()

huellas = {k: sha_archivo(p) for k, p in ENTRADAS.items()}
catalogo = pd.read_csv(ENTRADAS["catalogo"], dtype=str, keep_default_na=False)
consultas = pd.read_csv(ENTRADAS["consultas"], dtype=str, keep_default_na=False)
bm25, densa, hibrida = [json.loads(ENTRADAS[k].read_text(encoding="utf-8"))
                        for k in ("bm25", "densa", "hibrida")]
print("Entradas cargadas. Sin inferencias ni modificaciones del catálogo.")

Entradas cargadas. Sin inferencias ni modificaciones del catálogo.


## 2. Comprobar los 23 sistemas

In [2]:
verificar(bm25["version_esquema"] == "BM25_RESULTADOS_V1", "Esquema BM25 distinto.")
verificar(densa["version_esquema"] == "RECUPERACION_DENSA_V1", "Esquema denso distinto.")
verificar(hibrida["version_esquema"] == "RECUPERACION_HIBRIDA_RRF_V1", "Esquema RRF distinto.")
verificar(set(densa["sistemas"]) == MODELOS, "Faltan modelos densos.")
verificar(set(hibrida["sistemas"]) == {f"rrf_bm25_{m}" for m in MODELOS}, "Faltan híbridos.")
for k in ("catalogo", "consultas"):
    verificar(huellas[k] == densa["fuentes"][f"sha256_{k}"]
              == hibrida["fuentes"][f"sha256_{k}"] == bm25["fuentes"][k]["sha256"],
              f"Cambió la versión de {k}.")
for k in ("bm25", "densa"):
    verificar(huellas[k] == hibrida["archivos_entrada"][k]["sha256"], f"Entrada RRF distinta: {k}.")

campos_q = ["query_id", "necesidad_id", "division", "idioma", "consulta"]
q = consultas[campos_q].to_dict("records")
qids = [r["query_id"] for r in q]
verificar(len(qids) == len(set(qids)) == 24, "Se esperan 24 consultas únicas.")
verificar(q == densa["consultas"] == hibrida["consultas"], "Consultas diferentes.")
verificar([{k: r[k] for k in campos_q} for r in bm25["resultados"]] == q, "Consultas BM25 distintas.")
ids = set(catalogo["article_id"])
verificar(len(catalogo) == len(ids) == len({int(i) for i in ids}) == 519, "Catálogo inválido.")
verificar(catalogo[["titulo", "abstract"]].apply(lambda s: s.str.strip().ne("")).all().all(),
          "Falta título o resumen.")
verificar(sha_json(bm25["resultados"]) == bm25["sha256_resultados"], "BM25 alterado.")

runs = {"bm25": bm25["resultados"]}
for grupo in (densa, hibrida):
    for nombre, s in grupo["sistemas"].items():
        verificar(sha_json(s["resultados"]) == s["sha256_resultados"], f"Resultado alterado: {nombre}.")
        runs[nombre] = s["resultados"]
for nombre, resultados in runs.items():
    verificar([r["query_id"] for r in resultados] == qids, f"Consultas incompletas: {nombre}.")
    campo = "score_bm25" if nombre == "bm25" else ("score_rrf" if nombre.startswith("rrf_") else "score_coseno")
    for r in resultados:
        docs = r["documentos"]
        verificar(len(docs) == 519 and {x["article_id"] for x in docs} == ids, "Identificadores diferentes.")
        verificar(all(math.isfinite(x[campo]) for x in docs), "Puntuación no finita.")
        verificar(docs == sorted(docs, key=lambda x: (-x[campo], int(x["article_id"]))), "Orden inválido.")
        for posicion, doc in enumerate(docs, 1):
            rango_ok = doc[campo] >= 0 if nombre == "bm25" else (doc[campo] > 0 if campo == "score_rrf" else abs(doc[campo]) <= 1.00001)
            verificar(rango_ok, "Puntuación fuera de rango.")
            rank = None if nombre == "bm25" and doc[campo] == 0 else posicion
            verificar(doc["rank"] == rank, "Posición inválida.")

cfg = hibrida["configuracion"]
verificar(cfg["c"] == 60 and cfg["pesos"] == {"bm25": 1, "densa": 1}, "Condición RRF diferente.")
mapas = {nombre: {r["query_id"]: {x["article_id"]: x["rank"] for x in r["documentos"]}
                  for r in resultados} for nombre, resultados in runs.items()}
for nombre, s in hibrida["sistemas"].items():
    origen = densa["sistemas"][s["codificador"]]
    verificar(s["sha256_resultados_densos"] == origen["sha256_resultados"]
              and s["sha256_resultados_bm25"] == bm25["sha256_resultados"], "Origen RRF diferente.")
    for r in s["resultados"]:
        for doc in r["documentos"]:
            aid, qid = doc["article_id"], r["query_id"]
            rb, rd = mapas["bm25"][qid][aid], mapas[s["codificador"]][qid][aid]
            esperado = 1 / (60 + rd) + (0 if rb is None else 1 / (60 + rb))
            verificar(doc["rank_bm25"] == rb and doc["rank_denso"] == rd
                      and math.isclose(doc["score_rrf"], esperado, rel_tol=0, abs_tol=1e-15),
                      "La fusión no corresponde a las listas originales.")
print("Validados: 23 sistemas, 24 consultas y 137 016 puntuaciones RRF.")

Validados: 23 sistemas, 24 consultas y 137 016 puntuaciones RRF.


## 3. Construir el conjunto común

In [3]:
necesidades = {}
for nid, grupo in consultas.groupby("necesidad_id", sort=True):
    verificar(len(grupo) == 2 and set(grupo["idioma"]) == {"es", "en"}, "Falta una versión lingüística.")
    for k in ("division", "criterio_grado_0", "criterio_grado_1", "criterio_grado_2", "campos_para_juicio"):
        verificar(grupo[k].nunique() == 1, f"Criterio ES/EN diferente: {nid}, {k}.")
    r = grupo.iloc[0]
    verificar(r["division"] == ("desarrollo" if int(nid[1:]) <= 4 else "comparacion"), "División cambiada.")
    necesidades[nid] = {"division": r["division"],
        "consultas": {r["idioma"]: r["consulta"] for _, r in grupo.iterrows()},
        "criterios": {str(g): r[f"criterio_grado_{g}"] for g in (0, 1, 2)}}
verificar(len(necesidades) == 12, "Se esperan 12 necesidades.")

q_a_n = {r["query_id"]: r["necesidad_id"] for r in q}
pools = {nid: set() for nid in necesidades}
for resultados in runs.values():
    for r in resultados:
        pools[q_a_n[r["query_id"]]].update(doc["article_id"] for doc in r["documentos"]
            if doc["rank"] is not None and doc["rank"] <= K_POOL)

casos, plan, filas = [], {}, []
for nid, pool in pools.items():
    fuera = sorted(ids - pool, key=int)
    rng = random.Random(f"muestra:{SEMILLA}:{nid}")
    extra = rng.sample(fuera, min(EXTRAS_POR_NECESIDAD, len(fuera)))
    seleccion = sorted(pool | set(extra), key=int)
    random.Random(f"presentacion:{SEMILLA}:{nid}").shuffle(seleccion)
    casos.extend({"caso_id": f"{nid}-A{int(aid):04d}", "necesidad_id": nid, "article_id": aid}
                 for aid in seleccion)
    plan[nid] = {"ids_top20": sorted(pool, key=int), "ids_aleatorios_fuera": sorted(extra, key=int),
                 "n_fuera_del_pool": len(fuera), "n_no_seleccionados": len(fuera) - len(extra)}
    filas.append({"necesidad_id": nid, "division": necesidades[nid]["division"],
                  "top20_unicos": len(pool), "muestra_adicional": len(extra), "total": len(seleccion)})

seleccionados = {c["article_id"] for c in casos}
documentos = {aid: {"titulo": r["titulo"], "abstract": r["abstract"]}
              for aid, r in catalogo.set_index("article_id").iterrows() if aid in seleccionados}
verificar(len(casos) == len({(c["necesidad_id"], c["article_id"]) for c in casos}), "Hay pares repetidos.")
verificar(all({doc["article_id"] for doc in r["documentos"]
               if doc["rank"] is not None and doc["rank"] <= K_POOL}
              <= {c["article_id"] for c in casos if c["necesidad_id"] == q_a_n[r["query_id"]]}
              for resultados in runs.values() for r in resultados), "Cobertura top20 incompleta.")

base = {"version_esquema": "CONJUNTO_PERTINENCIA_V1",
    "protocolo": {"profundidad_pool": K_POOL, "extra_por_necesidad": EXTRAS_POR_NECESIDAD,
        "semilla": SEMILLA, "unidad_juicio": "necesidad_id + article_id",
        "campos_para_juicio": ["titulo", "abstract"], "idiomas_agrupados": ["es", "en"],
        "grados": {"2": "aporte directo", "1": "aporte parcial", "0": "sin aporte pertinente"},
        "grado_null": "pendiente o información insuficiente; nunca equivale a 0",
        "instrucciones": [
            "Aplicar la rúbrica de la necesidad; no exigir que un solo artículo conteste toda la consulta.",
            "Utilizar únicamente título y abstract para este juicio; no completar omisiones por inferencia.",
            "Ante información insuficiente, mantener grado null y explicar la duda.",
            "Registrar una justificación breve y fragmentos literales del título o abstract.",
            "No usar puntajes, posiciones, modelo recuperador ni condición de artículo orientador.",
            "Conservar por separado anotaciones asistidas y humanas; no llamar humanas a las de IA.",
            "No presentar el conjunto como exhaustivo ni como evaluación de predicciones de coautoría."],
        "regla_revision_fuera_pool": "Si la muestra aporta relevantes, ampliar la revisión de esa necesidad; cero hallazgos no demuestra exhaustividad.",
        "metricas_calculadas": False, "anotacion_automatica_ejecutada": False},
    "necesidades": necesidades, "documentos": documentos, "casos": casos,
    "control_no_mostrar_al_evaluador": {
        "archivos_entrada": {k: {"ruta": str(p.relative_to(REPO)).replace(chr(92), "/"), "sha256": huellas[k]}
                             for k, p in ENTRADAS.items()},
        "sistemas_incluidos": sorted(runs), "seleccion": plan,
        "resumen_por_necesidad": filas,
        "observaciones_previas_no_usar_como_etiquetas": consultas[consultas["idioma"].eq("es")][
            ["necesidad_id", "observaciones_revision"]].to_dict("records"),
        "advertencias_heredadas": hibrida.get("advertencias", []),
        "notas": ["No se fusionaron registros con distinto article_id, aunque compartan resumen.",
                  "Las observaciones previas contienen orientadores: no enviarlas al evaluador.",
                  "La muestra extra es exploratoria, no una garantía de recall completo."],
        "referencias_metodologicas": [
            "https://trec.nist.gov/howto.html",
            "https://www.nist.gov/publications/retrieval-evaluation-incomplete-information",
            "https://www.nist.gov/publications/comparison-pooled-and-sampled-relevance-judgments-trec-2006-terabyte-track"]}}
firma = sha_json(base)
print(f"{sum(len(p) for p in pools.values()):,} pares del top20 + {sum(len(p['ids_aleatorios_fuera']) for p in plan.values())} adicionales = {len(casos):,} casos.")

1,375 pares del top20 + 120 adicionales = 1,495 casos.


## 4. Guardar sin sobrescribir anotaciones

In [4]:
def juicio_vacio():
    return {"estado": "pendiente", "grado": None, "justificacion": "", "evidencia_literal": [],
            "evaluador": None, "fecha_utc": None}

for k, p in ENTRADAS.items():
    verificar(sha_archivo(p) == huellas[k], f"Cambió {k} durante la ejecución.")
if SALIDA.exists():
    conjunto = json.loads(SALIDA.read_text(encoding="utf-8"))
    verificar(conjunto.get("firma_conjunto") == firma and conjunto.get("base") == base,
              "Ya existe un conjunto diferente: no se sobrescribe.")
    verificar(set(conjunto.get("anotaciones", {})) == {c["caso_id"] for c in casos},
              "Las anotaciones no corresponden a los casos.")
    print("Conjunto existente: se conservan todas las anotaciones, sin escribir el archivo.")
else:
    conjunto = {"firma_conjunto": firma, "base": base,
                "anotaciones": {c["caso_id"]: {"asistida": juicio_vacio(), "humana": juicio_vacio()}
                                for c in casos}}
    temporal = SALIDA.with_suffix(".json.tmp")
    verificar(not temporal.exists(), "Hay una escritura temporal previa; revísala antes de continuar.")
    with temporal.open("x", encoding="utf-8") as f:
        json.dump(conjunto, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.write("\n")
    verificar(not SALIDA.exists(), "Apareció otra salida; se conserva el temporal.")
    temporal.rename(SALIDA)
    print("Guardado:", SALIDA)

def ver_caso(necesidad_id="N01", posicion=0):
    candidatos = [c for c in conjunto["base"]["casos"] if c["necesidad_id"] == necesidad_id]
    if not 0 <= posicion < len(candidatos):
        raise ValueError("Posición fuera del conjunto.")
    c = candidatos[posicion]
    n = conjunto["base"]["necesidades"][necesidad_id]
    doc = conjunto["base"]["documentos"][c["article_id"]]
    print(c["caso_id"], "|", n["consultas"]["es"], "\n", n["consultas"]["en"])
    for grado in ("2", "1", "0"):
        print(f"\nGrado {grado}: {n['criterios'][grado]}")
    print("\nTítulo:", doc["titulo"], "\n\nAbstract:", doc["abstract"])

resumen = pd.DataFrame(filas)
display(resumen)
print("Casos:", len(casos), "| Artículos distintos incluidos:", len(documentos))
print("Pendientes asistidos:", sum(v["asistida"]["estado"] == "pendiente" for v in conjunto["anotaciones"].values()))
print("Pendientes humanos:", sum(v["humana"]["estado"] == "pendiente" for v in conjunto["anotaciones"].values()))
print("No se asignaron etiquetas ni se calcularon métricas. Vista sin rankings: ver_caso('N01', 0)")

Guardado: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\conjunto_pertinencia_v1.json


,necesidad_id,division,top20_unicos,muestra_adicional,total
0,N01,desarrollo,119,10,129
1,N02,desarrollo,97,10,107
2,N03,desarrollo,143,10,153
3,N04,desarrollo,103,10,113
4,N05,comparacion,111,10,121
5,N06,comparacion,122,10,132
6,N07,comparacion,91,10,101
7,N08,comparacion,117,10,127
8,N09,comparacion,132,10,142
9,N10,comparacion,125,10,135


Casos: 1495 | Artículos distintos incluidos: 472
Pendientes asistidos: 1495
Pendientes humanos: 1495
No se asignaron etiquetas ni se calcularon métricas. Vista sin rankings: ver_caso('N01', 0)
